# 09 — RQ2: Forecastability of register demand from its own history

**Question.** Can regional register demand be forecast from its own recent history more accurately than a naive baseline, and over what horizon does that accuracy hold?

Thin driver: writes `src/rq2.py`, runs the expanding-window walk-forward, displays the RMSE/MAE/skill tables, commits + pushes. Run top to bottom.

### 1. Dependencies (already present in Colab: pandas, numpy, pyarrow)

In [1]:
!pip -q install pyarrow  # no-op if present

### 2. Mount Drive, enter the repo, pull first

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/capstone-project
!git pull --no-edit

Mounted at /content/drive
/content/drive/MyDrive/capstone-project
From https://github.com/fashdeen/capstone-project
   c0055fb..745e206  main       -> origin/main
Already up to date.


### 3. Author the RQ2 module
This `%%writefile` cell creates/overwrites `src/rq2.py`.

In [3]:
%%writefile src/rq2.py
"""
rq2.py — RQ2 forecastability of regional register demand from its own recent history.

Question
--------
Can regional register demand be forecast from its own recent history more accurately
than a naive baseline, and over what horizon does that accuracy hold?

Design (settled in supervision)
-------------------------------
  target      : demand_rate (level, per 1,000) — forecast h quarters ahead
  baseline    : naive random walk = carry last level forward (predict no change).
                Also reported vs a random-walk-WITH-drift (harder bar for a
                trending series): last level + h * the TA's mean historical change.
  forecaster  : register-only. AR(p) on the STATIONARY change series with TA
                effects (drift) and NO time effects (a future quarter's time
                effect is unknowable -> would leak the future). Iterated h steps
                and cumulated back to a level. p = 1 primary, p = 2 tested.
  validation  : expanding-window walk-forward. First forecast origin at t0 (~q20)
                so the model has enough history; forecast every TA at each origin;
                horizons 1..4. Only past data is used to fit each forecast.
  metrics     : out-of-sample RMSE and MAE per horizon, and skill = 1 - RMSE_model
                / RMSE_baseline (>0 means the model beats the baseline).

RQ3 hook: fe_ar_fit / fe_ar_forecast are written so market-signal columns can be
added as exogenous regressors without reworking the engine.
Framing is prediction, never causation.
"""
import numpy as np
import pandas as pd

import config

LEVEL, CHANGE = "demand_rate", "demand_rate_change"
ENTITY, TIME = "ta_key", "q"


# ───────────────────────── data grids ─────────────────────────
def load_panel():
    return pd.read_parquet(config.DATA_PROCESSED / "panel.parquet")


def make_grids(panel):
    """Return level grid L, change grid C (both ti x TA), and the quarter list."""
    qs = sorted(panel[TIME].unique())
    qidx = {q: i for i, q in enumerate(qs)}
    p = panel.copy()
    p["ti"] = p[TIME].map(qidx)
    L = p.pivot(index="ti", columns=ENTITY, values=LEVEL).sort_index()
    C = p.pivot(index="ti", columns=ENTITY, values=CHANGE).sort_index()
    return L, C, qs


# ───────────────────────── register-only AR(p) with TA effects ─────────────────────────
def fe_ar_fit(C, tas, plags):
    """Fixed-effects AR(plags) on the change series. Returns (phi, c_i) or None."""
    rows = []
    for i in tas:
        s = C[i].dropna()
        sidx = set(s.index)
        for t in s.index:
            if all((t - l) in sidx for l in range(1, plags + 1)):
                rows.append([i, s[t]] + [s[t - l] for l in range(1, plags + 1)])
    if len(rows) < 50:
        return None
    df = pd.DataFrame(rows, columns=["i", "y"] + [f"x{l}" for l in range(1, plags + 1)])
    for c in ["y"] + [f"x{l}" for l in range(1, plags + 1)]:
        df[c + "_d"] = df[c] - df.groupby("i")[c].transform("mean")   # within-TA demean = TA effects
    Xd = df[[f"x{l}_d" for l in range(1, plags + 1)]].values
    yd = df["y_d"].values
    phi, *_ = np.linalg.lstsq(Xd, yd, rcond=None)
    ci = {i: g["y"].mean() - sum(phi[l - 1] * g[f"x{l}"].mean() for l in range(1, plags + 1))
          for i, g in df.groupby("i")}
    return phi, ci


def fe_ar_forecast(i, t, L, C, phi, ci, plags, h):
    """Iterate the AR(p) h steps from origin t and cumulate to a level forecast."""
    hist = [C[i][t - l] if (t - l) >= 0 else np.nan for l in range(plags)]
    if any(np.isnan(hist)) or i not in ci:
        return np.nan
    ds = []
    for _ in range(h):
        d = ci[i] + sum(phi[l] * hist[l] for l in range(plags))
        ds.append(d)
        hist = [d] + hist[:-1]
    return L[i][t] + np.sum(ds)


# ───────────────────────── walk-forward + scoring ─────────────────────────
def walk_forward(L, C, t0=20, horizons=(1, 2, 3, 4), ar_lags=(1, 2)):
    T = L.shape[0]
    tas = list(L.columns)
    models = ["rw_flat", "rw_drift"] + [f"ar{p}_fe" for p in ar_lags]
    err = {m: {h: {"se": [], "ae": []} for h in horizons} for m in models}
    for t in range(t0, T):
        fits = {p: fe_ar_fit(C.loc[:t], tas, p) for p in ar_lags}
        for i in tas:
            Lt = L[i][t]
            if np.isnan(Lt):
                continue
            drift = C[i].loc[:t].mean()
            for h in horizons:
                if t + h >= T:
                    continue
                a = L[i][t + h]
                if np.isnan(a):
                    continue
                fc = {"rw_flat": Lt, "rw_drift": Lt + h * drift}
                for p in ar_lags:
                    fc[f"ar{p}_fe"] = (fe_ar_forecast(i, t, L, C, *fits[p], p, h)
                                       if fits[p] else np.nan)
                for m in models:
                    if not np.isnan(fc[m]):
                        err[m][h]["se"].append((a - fc[m]) ** 2)
                        err[m][h]["ae"].append(abs(a - fc[m]))
    return err, models


def score(err, models, horizons=(1, 2, 3, 4), baseline="rw_flat"):
    rmse = lambda m, h: np.sqrt(np.mean(err[m][h]["se"])) if err[m][h]["se"] else np.nan
    mae = lambda m, h: np.mean(err[m][h]["ae"]) if err[m][h]["ae"] else np.nan
    idx = [f"h{h}" for h in horizons]
    R = pd.DataFrame({m: [rmse(m, h) for h in horizons] for m in models}, index=idx).T
    M = pd.DataFrame({m: [mae(m, h) for h in horizons] for m in models}, index=idx).T
    S = pd.DataFrame({m: [1 - rmse(m, h) / rmse(baseline, h) for h in horizons]
                      for m in models if m != baseline}, index=idx).T
    n = {f"h{h}": len(err[baseline][h]["se"]) for h in horizons}
    return {"rmse": R.round(4), "mae": M.round(4), f"skill_vs_{baseline}": S.round(3), "n_test": n}


def run(panel=None, t0=20):
    panel = load_panel() if panel is None else panel
    L, C, qs = make_grids(panel)
    err, models = walk_forward(L, C, t0=t0)
    out = {"t0": t0, "origin_quarter": qs[t0], "err": err, "models": models}
    out.update(score(err, models, baseline="rw_flat"))
    out["skill_vs_rw_drift"] = score(err, models, baseline="rw_drift")["skill_vs_rw_drift"]
    return out


def print_report(out=None):
    out = run() if out is None else out
    print(f"RQ2 — forecastability: expanding-window walk-forward, first forecast at {out['origin_quarter']}")
    print("Target: demand_rate (level, per 1,000). Errors are out-of-sample, pooled over TAs.\n")
    print("RMSE by horizon:")
    print(out["rmse"].to_string(), "\n")
    print("MAE by horizon:")
    print(out["mae"].to_string(), "\n")
    print("Skill vs naive random walk  (1 - RMSE_model/RMSE_rw_flat; >0 beats naive):")
    print(out["skill_vs_rw_flat"].to_string(), "\n")
    print("Skill vs random-walk-with-drift (harder bar for a trending series):")
    print(out["skill_vs_rw_drift"].to_string(), "\n")
    print("Out-of-sample test points per horizon:", out["n_test"])
    return out


Writing src/rq2.py


### 4. Run RQ2 and display the result
Baselines = naive random walk (no change) and random-walk-with-drift. Forecaster = register-only AR(1)/AR(2) on the change, TA effects, no time effects. Skill > 0 means the model beats the baseline.

In [4]:
import sys, importlib, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')
import rq2; importlib.reload(rq2)

out = rq2.print_report()

RQ2 — forecastability: expanding-window walk-forward, first forecast at 2021Q1
Target: demand_rate (level, per 1,000). Errors are out-of-sample, pooled over TAs.

RMSE by horizon:
              h1      h2      h3      h4
rw_flat   0.4601  0.6717  0.8415  0.9641
rw_drift  0.5088  0.8154  1.1045  1.3809
ar1_fe    0.5073  0.8137  1.1067  1.3883
ar2_fe    0.5053  0.8071  1.0970  1.3795 

MAE by horizon:
              h1      h2      h3      h4
rw_flat   0.3213  0.4806  0.5987  0.6867
rw_drift  0.3571  0.5879  0.8064  1.0246
ar1_fe    0.3543  0.5860  0.8080  1.0330
ar2_fe    0.3521  0.5798  0.7982  1.0260 

Skill vs naive random walk  (1 - RMSE_model/RMSE_rw_flat; >0 beats naive):
             h1     h2     h3     h4
rw_drift -0.106 -0.214 -0.313 -0.432
ar1_fe   -0.103 -0.211 -0.315 -0.440
ar2_fe   -0.098 -0.202 -0.304 -0.431 

Skill vs random-walk-with-drift (harder bar for a trending series):
            h1     h2     h3     h4
rw_flat  0.096  0.176  0.238  0.302
ar1_fe   0.003  0.002 -0.

### 5. Inspect individual pieces (optional)

In [ ]:
out['rmse']            # RMSE by horizon

In [ ]:
out['skill_vs_rw_flat']  # skill vs the naive baseline

In [ ]:
out['n_test']          # out-of-sample test points per horizon

### 6. Commit & push
**Save the notebook first** (Ctrl/Cmd-S) so the committed `.ipynb` includes outputs. The `%cd` re-enters the repo so git runs in the right place.

In [ ]:
%cd /content/drive/MyDrive/capstone-project

from google.colab import userdata
tok = userdata.get('GH_TOKEN')

!git config user.name  'fashdeen'
!git config user.email 'fashdeen@yahoo.com'

!git add src/rq2.py notebooks/09_rq2.ipynb
!git commit -m 'RQ2: walk-forward forecastability (register-only vs naive baselines)'
!git push https://{tok}@github.com/fashdeen/capstone-project.git